In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('balanced.csv')

In [3]:
df.sample(5)

,Suspicious_Report,Sex,MaritalStatus,RepNumber,Age,High_Risk_Individual,FraudFound_P
378,1,Male,Married,1,50,0,0
996,1,Male,Married,9,46,0,1
355,0,Male,Married,3,32,0,0
553,1,Male,Single,5,36,0,0
440,0,Male,Married,16,55,0,0


In [4]:
col =['Sex','MaritalStatus']
ct = ColumnTransformer([('oe',OrdinalEncoder(),col)],remainder='passthrough')

In [5]:
x = df.drop('FraudFound_P',axis=1)
x = ct.fit_transform(x)
y = df['FraudFound_P']

In [6]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=34,shuffle=True)

In [7]:
def test(model):
    y_pred = model.predict(x_test)
    print('accuracy',accuracy_score(y_test,y_pred))
    print('precision',precision_score(y_test,y_pred))
    print('recall',recall_score(y_test,y_pred))
    print('f1',f1_score(y_test,y_pred))
    print(confusion_matrix(y_test,y_pred))

In [8]:
par_lr = {
    'penalty': ['l2', 'l1'],
    'C': [0.001, 0.01, 0.1, 1, 10, 100]
}
lr = LogisticRegression(solver='liblinear')
lr_cv = GridSearchCV(lr, par_lr, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
lr_cv.fit(x_train,y_train)
print(lr_cv.best_params_,lr_cv.best_score_)
test(lr_cv.best_estimator_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
{'C': 0.1, 'penalty': 'l1'} 0.7193746312684366
accuracy 0.7092198581560284
precision 0.6382978723404256
recall 0.8955223880597015
f1 0.7453416149068323
[[ 80  68]
 [ 14 120]]


c:\Users\kumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\kumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [9]:
par_rf = {
    'n_estimators': [10,100,1000],
    'max_depth': [2,3,4,5],
    'min_samples_split': [2,3,4],
    'min_samples_leaf': [1,2,3]
}
rf = RandomForestClassifier()
rf_cv = GridSearchCV(rf, par_rf, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
rf_cv.fit(x_train,y_train)
print(rf_cv.best_params_,rf_cv.best_score_)
test(rf_cv.best_estimator_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
{'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 10} 0.7184936086529006
accuracy 0.7304964539007093
precision 0.6611111111111111
recall 0.8880597014925373
f1 0.7579617834394905
[[ 87  61]
 [ 15 119]]


In [10]:
par_xgb = {
    'n_estimators': [10,100,400,700,1000],
    'max_depth': [2,3,4,5],
    'learning_rate': [0.01,0.1,1]
}
xgb = XGBClassifier()
xgb_cv = GridSearchCV(xgb, par_xgb, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
xgb_cv.fit(x_train,y_train)
print(xgb_cv.best_params_,xgb_cv.best_score_)
test(xgb_cv.best_estimator_)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 700} 0.7149419862340217
accuracy 0.7269503546099291
precision 0.6727272727272727
recall 0.8283582089552238
f1 0.7424749163879598
[[ 94  54]
 [ 23 111]]


In [11]:
par_scv = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
scv = SVC()
scv_cv = GridSearchCV(scv, par_scv, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
scv_cv.fit(x_train,y_train)
print(scv_cv.best_params_,scv_cv.best_score_)
test(scv_cv.best_estimator_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
{'C': 1, 'gamma': 'scale', 'kernel': 'linear'} 0.7140491642084562
accuracy 0.7269503546099291
precision 0.6792452830188679
recall 0.8059701492537313
f1 0.7372013651877133
[[ 97  51]
 [ 26 108]]


In [12]:
par_knn = {
    'n_neighbors': [3,5,7,9],
    'weights': ['uniform', 'distance']
}
knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, par_knn, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
knn_cv.fit(x_train,y_train)
print(knn_cv.best_params_,knn_cv.best_score_)
test(knn_cv.best_estimator_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'n_neighbors': 5, 'weights': 'uniform'} 0.5825722713864308
accuracy 0.5602836879432624
precision 0.5284090909090909
recall 0.6940298507462687
f1 0.6
[[65 83]
 [41 93]]


In [13]:
par_nb = {
    'var_smoothing': [0.001, 0.01, 0.1, 1, 10, 100]
}
nb = GaussianNB()
nb_cv = GridSearchCV(nb, par_nb, cv=5, n_jobs=-1, verbose=1, scoring='accuracy')
nb_cv.fit(x_train,y_train)
print(nb_cv.best_params_,nb_cv.best_score_)
test(nb_cv.best_estimator_)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
{'var_smoothing': 0.001} 0.7087197640117994
accuracy 0.6950354609929078
precision 0.6188118811881188
recall 0.9328358208955224
f1 0.7440476190476191
[[ 71  77]
 [  9 125]]


In [14]:
# nb has highest recall score it means it does not misses values of the fraud cases
# best model are svm xgb and random forest